# Fine-tune Laya on support triage (Colab)

Trains two models from the `english` Laya checkpoint on `train.jsonl` (3,800 messages × 6 questions) and scores them on `val.jsonl` (300):

| Model | What trains | Output |
| --- | --- | --- |
| `laya-ft-head` | final layer only (`--freeze-encoder`) | `models/laya-ft-head` |
| `laya-ft-full` | all 421M parameters | `models/laya-ft-full` |

**Before running:** Runtime → Change runtime type → **T4 GPU**. Upload `data/train.jsonl` and `data/val.jsonl` from the repo to the Drive folder `MyDrive/laya-vs-jev/` (cell 3 can upload them for you instead).

Everything is written to Drive, so a disconnect doesn't lose finished runs. Keep the tab open: free Colab disconnects idle sessions. See `docs/training.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install "laya==0.4.17"
!laya-train --help | head -5

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/laya-vs-jev"
os.makedirs(f"{WORK}/models", exist_ok=True)

missing = [f for f in ("train.jsonl", "val.jsonl") if not os.path.exists(f"{WORK}/{f}")]
if missing:
    from google.colab import files
    print("Upload:", missing)
    for name, data in files.upload().items():
        open(f"{WORK}/{name}", "wb").write(data)
!wc -l "{WORK}/train.jsonl" "{WORK}/val.jsonl"

In [ ]:
# Sanity check: every row becomes 6 valid items, no weights loaded
!laya-train --data "{WORK}/train.jsonl" --eval "{WORK}/val.jsonl" --base english --epochs 3 --dry-run

## 1. Final layer only (`laya-ft-head`)

Fast: the encoder is frozen, so this run mostly measures what the head alone can learn.

In [ ]:
%%time
!laya-train --data "{WORK}/train.jsonl" --eval "{WORK}/val.jsonl" --base english \
  --epochs 3 --freeze-encoder --device cuda --seed 0 \
  --out "{WORK}/models/laya-ft-head" 2>&1 | tee "{WORK}/models/laya-ft-head.log"

## 2. Full fine-tune (`laya-ft-full`)

Expect roughly 1–1.5 h per epoch on a T4, so about 3–4.5 h for 3 epochs. `micro_batch 8 × grad_accum 8` with AMP on CUDA, the upstream defaults.

In [ ]:
%%time
!laya-train --data "{WORK}/train.jsonl" --eval "{WORK}/val.jsonl" --base english \
  --epochs 3 --device cuda --seed 0 \
  --out "{WORK}/models/laya-ft-full" 2>&1 | tee "{WORK}/models/laya-ft-full.log"

## 3. Results and download

`train_report.json` in each model folder holds the before/after scores on `val.jsonl`. Download the two zips and unpack them into the repo's `models/` folder:

```bash
unzip laya-ft-head.zip -d models/ && unzip laya-ft-full.zip -d models/
```

In [ ]:
import json
for name in ("laya-ft-head", "laya-ft-full"):
    path = f"{WORK}/models/{name}/train_report.json"
    if os.path.exists(path):
        print(name, json.dumps(json.load(open(path)), indent=1)[:3000])

In [ ]:
%cd {WORK}/models
!for m in laya-ft-head laya-ft-full; do [ -d $m ] && zip -qr $m.zip $m && ls -lh $m.zip; done
from google.colab import files
for m in ("laya-ft-head", "laya-ft-full"):
    if os.path.exists(f"{m}.zip"):
        files.download(f"{m}.zip")